In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Load data
data = datasets.MNIST(
    "./data", train=True, download=True,
    transform=transforms.ToTensor()
)
loader = DataLoader(data, batch_size=128, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Define Autoencoder
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 20)
        )

        self.decoder = nn.Sequential(
            nn.Linear(20, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

model = Autoencoder().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

# 3. Train
for epoch in range(5):
    total_loss = 0

    for images, _ in loader:
        images = images.to(device)
        output = model(images)
        original = images.view(-1, 784)

        loss = criterion(output, original)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch + 1,
          "Loss:", total_loss / len(loader))

# 4. Display original and reconstructed images
model.eval()
images, _ = next(iter(loader))
images = images[:8].to(device)

with torch.no_grad():
    output = model(images)

plt.figure(figsize=(10, 3))
for i in range(8):
    plt.subplot(2, 8, i + 1)
    plt.imshow(images[i].cpu().squeeze(), cmap="gray")
    plt.axis("off")

    plt.subplot(2, 8, i + 9)
    plt.imshow(output[i].reshape(28, 28).cpu(), cmap="gray")
    plt.axis("off")
plt.show()